# Knowledge Graph Debugging and Case Inspection

Case-level inspection and debugging of knowledge graph construction and snapshots. Supports identification of structural inconsistencies, missing links, and pipeline errors, as well as targeted analysis of individual cases.

In [1]:
import json
import tarfile
import tempfile
import zlib

from pathlib import Path

import pygit2
import polars as pl

from tqdm import tqdm

from benchmark.definitions.partitions import Shard
from benchmark.attribution.history import VersionControl, VersionControlSnapshot
from benchmark.attribution.knowledge import KnowledgeGraphBuilder, KnowledgeGraphStore
from benchmark_notebook.io import CloudStorage
from benchmark_notebook.knowledge import KnowledgeGraphRenderer

table = pl.from_arrow(
    CloudStorage(bucket="bug-request-bucket").asset(
        key="ingest-validation-set",
        version="1.1.1",
        location="ingest",
        partition="default",
    )
)

In [2]:
names = sorted(table["repository"].unique().to_list())
name = Shard.from_space("db3412c", names)  # "arimah/condict"
partition = Shard.to_space(names, name)

## Build

In [3]:
archive_build = CloudStorage(bucket="bug-request-bucket").asset(
    key="ingest-validation-sources",
    version="1.0.1",
    location="ingest",
    partition=partition,
)

payload = table.filter(pl.col("repository") == name).row(0, named=True)
correction = payload["correction"]

work = Path(f"/tmp/tmp{zlib.crc32(str(archive_build).encode()):x}")

if not work.exists():
    work.mkdir(parents=True)

    with tarfile.open(archive_build, "r:gz") as tar:
        with tqdm(
            total=archive_build.stat().st_size,
            unit="B",
            unit_scale=True,
            desc="Extracting",
        ) as bar:
            for member in tar:
                tar.extract(member, path=work, filter="data")
                bar.update(member.size)

repository_build = VersionControl(pygit2.Repository(str(work / "repository.git")))
parents = repository_build.parents(correction)
base = correction if not parents else parents[0]
store_build = KnowledgeGraphBuilder(repository=repository_build).build(base, correction)
KnowledgeGraphRenderer().render(store_build)

## Snapshot

In [3]:
archive = CloudStorage(bucket="bug-request-bucket").asset(
    key="ingest-validation-context",
    version="3.1.3",
    location="main",
    partition=partition,
)

work = Path(f"/tmp/tmp{zlib.crc32(str(archive).encode()):x}")

if not work.exists():
    work.mkdir(parents=True)

    with tarfile.open(archive, "r:gz") as tar:
        with tqdm(
            total=archive.stat().st_size,
            unit="B",
            unit_scale=True,
            desc="Extracting",
        ) as bar:
            for member in tar:
                tar.extract(member, path=work, filter="data")
                bar.update(member.size)

repository_snapshot = VersionControlSnapshot.load(work / name / "snapshot")

for context in sorted((work / name / "context").glob("*.json")):
    with open(context, "r", encoding="utf-8") as file:
        store_snapshot = KnowledgeGraphStore.load(json.load(file))

store_snapshot.dump()
KnowledgeGraphRenderer().render(store_snapshot).show(
    config={
        "toImageButtonOptions": {
            "format": "svg",
            "filename": "plot",
            "width": 900,
            "scale": 1,
        }
    }
)